# 04-2. LLM 기반 출시 후 패치·운영 전략 제안 - 리포트 형식 출력 버전

04번에서 Steam 리뷰를 LLM으로 분류하고, 04-1번에서 분류 결과를 전처리했다.  
이 코드은 04-1에서 만든 근거 데이터를 바탕으로 **Tainted Grail: The Fall of Avalon**의 패치·운영 전략을 리포트 문장 형식으로 확인한다.

핵심 원칙은 다음과 같다.

- 우선 검토 수준은 LLM이 직접 판단하지 않는다.
- 우선 검토 수준과 대응 구분은 04-1에서 계산한 `rule_priority_hint`, `action_group_hint`를 그대로 사용한다.
- LLM은 이미 정리된 근거를 바탕으로 개발자가 이해하기 쉬운 전략 문장과 실행안을 만드는 역할만 한다.
- 이 버전은 파일 저장 없이 코드 화면에 보고서 문장 중심으로 출력한다.

# 0. 환경설정

In [ ]:
# ============================================================
# 기본 라이브러리
# ============================================================
import os
import json
import platform
from pathlib import Path
from datetime import datetime
from typing import List, Literal

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# ============================================================
# LLM / Pydantic 관련 라이브러리
# ============================================================
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 200)

# 1. 기본 설정

In [2]:
# ============================================================
# 프로젝트 경로 설정
# ============================================================
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정한다.
ROOT = Path.cwd()

# Jupyter 실행 위치가 하위 폴더일 수 있으므로,
# data/preprocessed 폴더가 보일 때까지 상위 폴더를 탐색한다.
if not (ROOT / "data" / "preprocessed").exists():
    for parent in ROOT.parents:
        if (parent / "data" / "preprocessed").exists():
            ROOT = parent
            break

# ============================================================
# 04_run_llm_postlaunch_tainted-grail_analysis.ipynb에서 사용한 실행 이름과 맞춘다.
# ============================================================
RUN_NAME = "postlaunch_tainted-grail"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME

# ============================================================
# 04-1 전처리 산출물 폴더
# ============================================================
POSTLAUNCH_PREPROCESS_DIR = OUTPUT_DIR / "postlaunch_preprocess_data"

REVIEW_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_review_base.csv"
ISSUE_SUMMARY_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_issue_summary.csv"
PATCH_OPS_EVIDENCE_BASE_PATH = POSTLAUNCH_PREPROCESS_DIR / "postlaunch_patch_ops_evidence_base.csv"
TABLEAU_POSTLAUNCH_SOURCE_PATH = POSTLAUNCH_PREPROCESS_DIR / "tableau_postlaunch_patch_ops_source.csv"

print("ROOT:", ROOT)
print("RUN_NAME:", RUN_NAME)
print("04-1 전처리 폴더:", POSTLAUNCH_PREPROCESS_DIR)
print("04-2 출력 방식: 코드 화면 출력 전용")


ROOT: c:\Users\joon5\Documents\github\steam-indie-game-analysis
RUN_NAME: postlaunch_tainted-grail
04-1 전처리 폴더: c:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\postlaunch_tainted-grail\postlaunch_preprocess_data
04-2 출력 방식: 코드 화면 출력 전용


In [3]:
# ============================================================
# LLM 실행 여부
# ============================================================
# False: LLM 호출 없이 근거 데이터와 프롬프트만 확인한다.
# True : 실제 LLM을 호출해서 패치·운영 전략 초안을 생성한다.

RUN_PATCH_OPS_LLM = True

# ============================================================
# 분석 대상 게임
# ============================================================
TARGET_APPID = 1466060
TARGET_GAME_NAME = "Tainted Grail: The Fall of Avalon"

# ============================================================
# 프롬프트에 넣을 근거 데이터 개수
# ============================================================
# 04-1에서 정리한 이슈 근거 중 상위 이슈를 LLM에게 제공한다.
# 최종 우선순위는 여기서 새로 판단하지 않고, 04-1의 rule_priority_hint를 그대로 사용한다.

MAX_ISSUES_FOR_PROMPT = 18
MAX_EVIDENCE_TEXT_ISSUES = 12

GROUP_LIMITS = {
    "즉시 확인": 5,
    "단기 개선": 6,
    "운영 커뮤니케이션 개선": 2,
    "장기 검토": 4,
    "강점 유지": 3,
    "검토 필요": 2,
}

# ============================================================
# LLM 호출 설정
# ============================================================
# temperature를 0으로 고정해 같은 근거에서 결과가 흔들리는 것을 줄인다.
MAX_RETRIES = 3
TEMPERATURE = 0.0

print("RUN_PATCH_OPS_LLM:", RUN_PATCH_OPS_LLM)
print("분석 대상 게임:", TARGET_GAME_NAME)
print("TARGET_APPID:", TARGET_APPID)
print("MAX_ISSUES_FOR_PROMPT:", MAX_ISSUES_FOR_PROMPT)


RUN_PATCH_OPS_LLM: True
분석 대상 게임: Tainted Grail: The Fall of Avalon
TARGET_APPID: 1466060
MAX_ISSUES_FOR_PROMPT: 18


# 2. Vertex AI / PydanticAI 설정

In [ ]:
# ============================================================
# Vertex AI 설정
# ============================================================
# 04_run_llm_postlaunch_tainted-grail_analysis.ipynb와 같은 방식으로 구성한다.

load_dotenv()
load_dotenv(ROOT / ".env")

GOOGLE_CLOUD_PROJECT = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GCP_PROJECT_ID")
)

GOOGLE_CLOUD_LOCATION = (
    os.getenv("GOOGLE_CLOUD_LOCATION")
    or os.getenv("VERTEX_LOCATION")
    or "global"
)

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")

if GOOGLE_CLOUD_PROJECT:
    vertex_provider = GoogleProvider(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
    )

    vertex_model = GoogleModel(
        GEMINI_MODEL,
        provider=vertex_provider,
    )

    print("Vertex AI project:", GOOGLE_CLOUD_PROJECT)
    print("Vertex AI location:", GOOGLE_CLOUD_LOCATION)
    print("Gemini model:", GEMINI_MODEL)
    print("Vertex 모델 생성: O")
else:
    vertex_provider = None
    vertex_model = None

    print("Vertex AI project: 미설정")
    print("Vertex 모델 생성: X")
    print("실제 LLM 실행 전 .env에 GOOGLE_CLOUD_PROJECT를 설정하세요.")

Vertex AI project: gen-lang-client-0587784564
Vertex AI location: global
Gemini model: gemini-3.1-flash-lite-preview
Vertex 모델 생성: O


# 3. 공통 함수

In [5]:
# ============================================================
# JSON 변환 보조 함수
# ============================================================

def to_serializable(obj):
    """Pydantic, pandas, numpy 값을 기본 Python 타입으로 바꾼다."""
    if isinstance(obj, BaseModel):
        return to_serializable(obj.model_dump())
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_serializable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, datetime):
        return obj.isoformat()
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


# ============================================================
# 프롬프트용 텍스트 표 생성 함수
# ============================================================

def shorten_text(value, max_chars=180):
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\n", " ").strip()
    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."
    return text


def df_to_text_table(df, columns, max_rows=20, max_cell_chars=180):
    """tabulate 없이 LLM 프롬프트에 넣을 간단 텍스트 표를 만든다."""
    if df is None or len(df) == 0:
        return "해당 조건에 맞는 근거 데이터가 없습니다."

    cols = [col for col in columns if col in df.columns]

    if len(cols) == 0:
        return "표시할 수 있는 근거 컬럼이 없습니다."

    small = df[cols].head(max_rows).copy()

    lines = []
    header = " | ".join(cols)
    lines.append(header)
    lines.append("-" * len(header))

    for _, row in small.iterrows():
        values = [shorten_text(row.get(col, ""), max_cell_chars) for col in cols]
        lines.append(" | ".join(values))

    return "\n".join(lines)


# ============================================================
# 정렬 보조 함수
# ============================================================

def add_order_columns(df):
    out = df.copy()
    priority_order_map = {"상": 1, "중": 2, "하": 3}
    action_order_map = {
        "즉시 확인": 1,
        "단기 개선": 2,
        "운영 커뮤니케이션 개선": 3,
        "장기 검토": 4,
        "검토 필요": 5,
        "강점 유지": 6,
    }

    out["priority_order"] = out.get("rule_priority_hint", "").map(priority_order_map).fillna(9)
    out["action_order"] = out.get("action_group_hint", "").map(action_order_map).fillna(9)
    return out


# 4. 데이터 불러오기

In [ ]:
# ============================================================
# 04-1에서 만든 CSV 불러오기
# ============================================================

review_base = pd.read_csv(REVIEW_BASE_PATH)
issue_summary = pd.read_csv(ISSUE_SUMMARY_PATH)
evidence_base = pd.read_csv(PATCH_OPS_EVIDENCE_BASE_PATH)
tableau_source = pd.read_csv(TABLEAU_POSTLAUNCH_SOURCE_PATH)

# 문자열 컬럼 정리
for df in [review_base, issue_summary, evidence_base, tableau_source]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# 날짜 컬럼 변환
for col in ["review_datetime", "release_date"]:
    if col in review_base.columns:
        review_base[col] = pd.to_datetime(review_base[col], errors="coerce")
    if col in tableau_source.columns:
        tableau_source[col] = pd.to_datetime(tableau_source[col], errors="coerce")

print("review_base:", review_base.shape)
print("issue_summary:", issue_summary.shape)
print("evidence_base:", evidence_base.shape)
print("tableau_source:", tableau_source.shape)

display(evidence_base.head())

review_base: (1000, 28)
issue_summary: (21, 26)
evidence_base: (21, 27)
tableau_source: (1967, 36)


C:\Users\joon5\AppData\Local\Temp\ipykernel_9172\3146103069.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\joon5\AppData\Local\Temp\ipykernel_9172\3146103069.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

,llm_issue_category,appid,game_name,issue_name_kor,affected_review_count,positive_review_count,negative_review_count,mixed_review_count,negative_mixed_review_count,steam_negative_review_count,high_urgency_review_count,high_urgency_negative_mixed_review_count,recent_30d_review_count,recent_30d_negative_mixed_review_count,early_playtime_negative_mixed_review_count,avg_playtime_at_review_hours,median_playtime_at_review_hours,high_urgency_rate,negative_mixed_rate,steam_negative_rate,recent_30d_negative_mixed_rate,action_group_hint,rule_priority_hint,priority_rule_detail,priority_reason,patch_ops_note,llm_evidence_text
0,bug,1466060,Tainted Grail: The Fall of Avalon,버그,95,4,85,0,85,41,38,38,38,33,11,44.86,34.28,0.4000,0.8947,0.4316,0.8684,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 95개, 부정·혼합 85개, Steam 비추천 맥락 41개, 최근 30일 부정·혼합 33개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 38개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락","반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.",[ISSUE]\nissue_category: bug\nissue_name_kor: 버그\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\npriority_rule_detail: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락\naffected_review_count: 95\nnegative...
1,performance,1466060,Tainted Grail: The Fall of Avalon,성능,60,5,53,0,53,26,26,26,25,22,13,26.13,19.30,0.4333,0.8833,0.4333,0.8800,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 60개, 부정·혼합 53개, Steam 비추천 맥락 26개, 최근 30일 부정·혼합 22개, 초반 플레이타임 부정·혼합 13개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락","프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.",[ISSUE]\nissue_category: performance\nissue_name_kor: 성능\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\npriority_rule_detail: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락\naffected_review_count: 60\n...
2,crash,1466060,Tainted Grail: The Fall of Avalon,크래시,30,0,29,0,29,25,26,26,12,11,7,28.43,20.27,0.8667,0.9667,0.8333,0.9167,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 30개, 부정·혼합 29개, Steam 비추천 맥락 25개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 7개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락","크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.",[ISSUE]\nissue_category: crash\nissue_name_kor: 크래시\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\npriority_rule_detail: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락\naffected_review_count: 30\nnegat...
3,optimization,1466060,Tainted Grail: The Fall of Avalon,최적화,27,1,25,1,26,16,14,14,11,11,11,19.36,8.40,0.5185,0.9630,0.5926,1.0000,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 27개, 부정·혼합 26개, Steam 비추천 맥락 16개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 14개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락",최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.,[ISSUE]\nissue_category: optimization\nissue_name_kor: 최적화\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\npriority_rule_detail: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락\naffected_review_count: 27...
4,save_progression,1466060,Tainted Grail: The Fall of Avalon,저장/진행,27,1,24,0,24,23,23,23,8,8,3,26.12,20.45,0.8519,0.8889,0.8519,1.0000,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,"영향 리뷰 27개, 부정·혼합 24개, Steam 비추천 맥락 23개, 최근 30일 부정·혼합 8개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 23개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락","저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.",[ISSUE]\nissue_category: save_progression\nissue_name_kor: 저장/진행\naction_group_hint: 즉시 확인\nrule_priority_hint: 상\npriority_rule_detail: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락\naffected_review_cou...


In [ ]:
# ============================================================
# 필수 컬럼 확인
# ============================================================

required_evidence_cols = [
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
    "priority_reason",
    "patch_ops_note",
    "llm_evidence_text",
]

missing_cols = [col for col in required_evidence_cols if col not in evidence_base.columns]
if missing_cols:
    raise ValueError(f"evidence_base에 필요한 컬럼이 없습니다: {missing_cols}")

print("필수 컬럼 확인 완료")

필수 컬럼 확인 완료


# 5. 분석 대상 요약 및 근거 데이터 선택

In [ ]:
# ============================================================
# 분석 대상 기본 요약
# ============================================================

def value_count_text(series):
    counts = series.value_counts(dropna=False)
    return ", ".join([f"{idx}: {cnt}" for idx, cnt in counts.items()])

analysis_overview = {
    "game_name": TARGET_GAME_NAME,
    "appid": TARGET_APPID,
    "review_count": int(len(review_base)),
    "issue_tag_count": int(len(tableau_source)),
    "review_date_min": review_base["review_datetime"].min().strftime("%Y-%m-%d") if "review_datetime" in review_base.columns else "",
    "review_date_max": review_base["review_datetime"].max().strftime("%Y-%m-%d") if "review_datetime" in review_base.columns else "",
    "steam_label_distribution": value_count_text(review_base["steam_label_text"]) if "steam_label_text" in review_base.columns else "",
    "llm_sentiment_distribution": value_count_text(review_base["llm_sentiment"]) if "llm_sentiment" in review_base.columns else "",
    "high_urgency_review_count": int(review_base.get("high_urgency_flag", pd.Series(dtype=bool)).sum()) if "high_urgency_flag" in review_base.columns else 0,
    "recent_30d_review_count": int(review_base.get("recent_30d_flag", pd.Series(dtype=bool)).sum()) if "recent_30d_flag" in review_base.columns else 0,
}

print(json.dumps(analysis_overview, ensure_ascii=False, indent=2))

{
  "game_name": "Tainted Grail: The Fall of Avalon",
  "appid": 1466060,
  "review_count": 1000,
  "issue_tag_count": 1967,
  "review_date_min": "2026-01-29",
  "review_date_max": "2026-04-29",
  "steam_label_distribution": "positive: 671, negative: 329",
  "llm_sentiment_distribution": "positive: 592, negative: 302, mixed: 103, neutral: 3",
  "high_urgency_review_count": 178,
  "recent_30d_review_count": 373
}


In [9]:
# ============================================================
# 04-2 LLM 입력용 근거 데이터 선택
# ============================================================
# 목적:
# - LLM에게 모든 원천 데이터를 주는 것이 아니라, 04-1에서 집계된 이슈별 근거를 준다.
# - 대응 구분별 대표 이슈를 골라 프롬프트가 너무 길어지는 것을 막는다.
# - 최종 우선순위는 여기서 새로 판단하지 않고, 04-1의 rule_priority_hint를 그대로 사용한다.
# - High urgency는 정렬의 핵심 기준으로 쓰지 않고 보조 지표로만 유지한다.

work = add_order_columns(evidence_base)

sort_cols = [
    "action_order",
    "priority_order",
    "negative_mixed_review_count",
    "steam_negative_review_count",
    "recent_30d_negative_mixed_review_count",
    "affected_review_count",
]

work = work.sort_values(sort_cols, ascending=[True, True, False, False, False, False])

selected_parts = []
for group_name, limit in GROUP_LIMITS.items():
    part = work[work["action_group_hint"] == group_name].head(limit)
    selected_parts.append(part)

selected_evidence = (
    pd.concat(selected_parts, ignore_index=True)
    .drop_duplicates("issue_name_kor")
    .head(MAX_ISSUES_FOR_PROMPT)
    .copy()
)

selected_evidence = add_order_columns(selected_evidence)
selected_evidence = selected_evidence.sort_values(sort_cols, ascending=[True, True, False, False, False, False])

print("선택된 이슈 근거 수:", len(selected_evidence))
print("04-1에서 계산한 이슈별 근거표는 마지막 출력부에서 다시 표 형태로 확인한다.")

선택된 이슈 근거 수: 18
04-1에서 계산한 이슈별 근거표는 마지막 출력부에서 다시 표 형태로 확인한다.


# 6. LLM 출력 스키마 정의

In [10]:
# ============================================================
# 패치·운영 전략 출력 스키마
# ============================================================
# priority와 action_group은 LLM이 새로 정하는 값이 아니다.
# 04-1에서 만든 rule_priority_hint, action_group_hint를 그대로 복사하는 값이다.

class PatchOpsItem(BaseModel):
    action_group: Literal["즉시 확인", "단기 개선", "운영 커뮤니케이션 개선", "장기 검토", "강점 유지", "검토 필요"] = Field(
        description="근거표의 action_group_hint를 그대로 복사한 대응 구분"
    )
    priority: Literal["상", "중", "하"] = Field(
        description="근거표의 rule_priority_hint를 그대로 복사한 고정 우선 검토 수준"
    )
    issue_name: str = Field(
        description="근거가 된 이슈명. 반드시 근거표의 issue_name_kor 값 중 하나를 그대로 사용"
    )
    patch_ops_direction: str = Field(
        description="해당 이슈에 대한 패치·운영 방향 요약"
    )
    detailed_actions: List[str] = Field(
        description="실제로 검토할 세부 실행안 2~4개"
    )
    evidence_summary: str = Field(
        description="제공된 근거표의 수치와 대표 리뷰 근거를 바탕으로 한 요약"
    )
    expected_effect: str = Field(
        description="이 대응이 기대하는 유저 경험 개선 효과"
    )
    caution: str = Field(
        description="해석이나 실행 시 주의해야 할 점"
    )


class PostLaunchPatchOpsResult(BaseModel):
    title: str = Field(description="보고서 제목")
    game_summary: str = Field(description="분석 대상 게임 요약")
    data_summary: str = Field(description="분석 데이터 규모와 기간 요약")
    current_status_summary: str = Field(description="최근 리뷰 기준 현재 반응 상태 요약")
    immediate_actions: List[PatchOpsItem] = Field(description="action_group_hint가 즉시 확인인 항목")
    short_term_improvements: List[PatchOpsItem] = Field(description="action_group_hint가 단기 개선인 항목")
    operation_communication: List[PatchOpsItem] = Field(description="action_group_hint가 운영 커뮤니케이션 개선인 항목")
    long_term_reviews: List[PatchOpsItem] = Field(description="action_group_hint가 장기 검토 또는 검토 필요인 항목")
    strengths_to_keep: List[PatchOpsItem] = Field(description="action_group_hint가 강점 유지인 항목")
    operation_notes: List[str] = Field(description="패치 노트, 커뮤니티 공지, 모니터링 등 운영 관점 제안")
    cautions: List[str] = Field(description="해석 시 주의사항")
    final_summary: str = Field(description="전체 패치·운영 방향 요약")


print("패치·운영 전략 출력 스키마 정의 완료")


패치·운영 전략 출력 스키마 정의 완료


# 7. LLM 프롬프트 생성

In [11]:
# ============================================================
# 프롬프트 생성 함수
# ============================================================

def build_patch_ops_prompt(analysis_overview, selected_evidence_df):
    selected_evidence_df = selected_evidence_df.copy()

    evidence_cols = [
        "issue_name_kor",
        "action_group_hint",
        "rule_priority_hint",
        "priority_rule_detail",
        "affected_review_count",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "recent_30d_negative_mixed_review_count",
        "early_playtime_negative_mixed_review_count",
        "high_urgency_review_count",
        "high_urgency_rate",
        "negative_mixed_rate",
        "steam_negative_rate",
        "priority_reason",
        "patch_ops_note",
    ]
    evidence_cols = [col for col in evidence_cols if col in selected_evidence_df.columns]

    evidence_text = df_to_text_table(
        selected_evidence_df,
        columns=evidence_cols,
        max_rows=MAX_ISSUES_FOR_PROMPT,
        max_cell_chars=180,
    )

    review_evidence_text = "\n\n".join(
        selected_evidence_df["llm_evidence_text"].head(MAX_EVIDENCE_TEXT_ISSUES).tolist()
    )

    prompt = f"""
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

목표:
{TARGET_GAME_NAME}의 최근 Steam 리뷰를 LLM으로 분류하고,
04-1에서 집계한 이슈별 반복성, 부정·혼합 분포, Steam 비추천 맥락, 최근성, 플레이타임 근거를 바탕으로
패치·운영 전략 초안을 작성하세요.

가장 중요한 제한:
- 당신은 상·중·하 우선 검토 수준을 새로 계산하거나 판단하지 않습니다.
- 근거표의 action_group_hint는 이미 04-1에서 데이터 기준으로 계산된 대응 구분입니다.
- 근거표의 rule_priority_hint는 이미 04-1에서 데이터 기준으로 계산된 고정 우선 검토 수준입니다.
- 각 항목의 action_group은 반드시 근거표의 action_group_hint를 그대로 사용하세요.
- 각 항목의 priority는 반드시 근거표의 rule_priority_hint를 그대로 사용하세요.
- action_group_hint를 바꾸거나, rule_priority_hint를 올리거나 내리지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- issue_name에는 반드시 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.
- High urgency는 04번 리뷰 분류 단계에서 LLM이 분류한 보조 지표입니다.
- High urgency를 근거로 우선 검토 수준을 새로 판단하거나, 우선 검토 수준을 올리지 마세요.

우선 검토 수준 해석 기준:
- rule_priority_hint는 04-1에서 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 부정 반응을 기준으로 계산되었습니다.
- priority_rule_detail은 rule_priority_hint가 부여된 규칙 설명입니다.
- priority_reason은 수치 근거를 사람이 읽기 쉽게 정리한 문장입니다.
- LLM은 이 값을 해석해 문장으로 풀어쓰되, 순위 자체를 바꾸지 않습니다.

근거 사용 기준:
- affected_review_count는 해당 이슈가 언급된 리뷰 수입니다.
- negative_mixed_review_count는 LLM이 부정 또는 혼합 맥락으로 분류한 리뷰 수입니다.
- steam_negative_review_count는 Steam 비추천 리뷰 수입니다.
- recent_30d_negative_mixed_review_count는 최근 30일에도 부정·혼합 이슈가 반복되는지 보는 기준입니다.
- early_playtime_negative_mixed_review_count는 짧은 플레이타임에서 부정·혼합 이슈가 나타나는지 보는 기준입니다.
- high_urgency_review_count는 04번 리뷰 분류 단계에서 LLM이 High urgency 후보로 분류한 리뷰 수이며 보조 참고 지표입니다.
- 수치가 작거나 근거가 약한 경우에는 단정하지 말고 "검토 필요" 또는 "추가 확인 필요"라고 표현하세요.

작성 기준:
- 패치·운영 제안은 개발자가 실제로 실행할 수 있는 문장으로 작성하세요.
- 즉시 확인 항목은 재현, 로그 확인, 진행 차단 여부, 크래시/성능/저장 문제 확인처럼 구체적으로 작성하세요.
- 단기 개선 항목은 경험 품질을 낮추는 반복 문제를 다음 패치에서 점검하는 방향으로 작성하세요.
- 운영 커뮤니케이션 개선 항목은 패치 노트, 공지, 커뮤니티 응답, 알려진 이슈 안내 관점으로 작성하세요.
- 장기 검토 항목은 개발 범위가 큰 콘텐츠, 스토리, 구조 개선을 로드맵 관점으로 작성하세요.
- 강점 유지 항목은 업데이트와 커뮤니티 메시지에서 계속 살릴 요소로 작성하세요.
- 리뷰 수나 LLM 분류만으로 실제 버그 원인을 확정하지 마세요.
- "반드시 개선된다", "성공한다" 같은 보장 표현은 쓰지 마세요.

분석 대상 요약:
{json.dumps(analysis_overview, ensure_ascii=False, indent=2)}

이슈별 집계 근거표:
{evidence_text}

대표 리뷰 근거와 LLM 개선 제안 후보:
{review_evidence_text}

출력 요구:
1. 게임과 데이터 규모를 간단히 요약하세요.
2. 현재 리뷰 상태를 2~3문장으로 요약하세요.
3. immediate_actions에는 action_group_hint가 "즉시 확인"인 항목만 작성하세요.
4. short_term_improvements에는 action_group_hint가 "단기 개선"인 항목만 작성하세요.
5. operation_communication에는 action_group_hint가 "운영 커뮤니케이션 개선"인 항목만 작성하세요.
6. long_term_reviews에는 action_group_hint가 "장기 검토" 또는 "검토 필요"인 항목만 작성하세요.
7. strengths_to_keep에는 action_group_hint가 "강점 유지"인 항목만 작성하세요.
8. 각 항목은 issue_name, priority, action_group, patch_ops_direction, detailed_actions, evidence_summary, expected_effect, caution을 포함해야 합니다.
9. evidence_summary에는 가능한 한 affected_review_count, negative_mixed_review_count, steam_negative_review_count, recent_30d_negative_mixed_review_count 중 2개 이상을 포함하세요.
10. operation_notes에는 패치 노트, 커뮤니티 공지, 모니터링 관점의 운영 제안을 작성하세요.
11. 마지막에는 해석 시 주의사항과 최종 요약을 작성하세요.
"""

    return prompt.strip()


patch_ops_prompt = build_patch_ops_prompt(
    analysis_overview=analysis_overview,
    selected_evidence_df=selected_evidence,
)

print("프롬프트 생성 완료")
print("프롬프트는 파일로 저장하지 않고 코드 화면에서만 확인합니다.")
print(patch_ops_prompt[:2500])


프롬프트 생성 완료
프롬프트는 파일로 저장하지 않고 코드 화면에서만 확인합니다.
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

목표:
Tainted Grail: The Fall of Avalon의 최근 Steam 리뷰를 LLM으로 분류하고,
04-1에서 집계한 이슈별 반복성, 부정·혼합 분포, Steam 비추천 맥락, 최근성, 플레이타임 근거를 바탕으로
패치·운영 전략 초안을 작성하세요.

가장 중요한 제한:
- 당신은 상·중·하 우선 검토 수준을 새로 계산하거나 판단하지 않습니다.
- 근거표의 action_group_hint는 이미 04-1에서 데이터 기준으로 계산된 대응 구분입니다.
- 근거표의 rule_priority_hint는 이미 04-1에서 데이터 기준으로 계산된 고정 우선 검토 수준입니다.
- 각 항목의 action_group은 반드시 근거표의 action_group_hint를 그대로 사용하세요.
- 각 항목의 priority는 반드시 근거표의 rule_priority_hint를 그대로 사용하세요.
- action_group_hint를 바꾸거나, rule_priority_hint를 올리거나 내리지 마세요.
- 근거표에 없는 issue_name_kor를 새로 만들지 마세요.
- issue_name에는 반드시 근거표에 있는 issue_name_kor 값을 그대로 작성하세요.
- High urgency는 04번 리뷰 분류 단계에서 LLM이 분류한 보조 지표입니다.
- High urgency를 근거로 우선 검토 수준을 새로 판단하거나, 우선 검토 수준을 올리지 마세요.

우선 검토 수준 해석 기준:
- rule_priority_hint는 04-1에서 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 부정 반응을 기준으로 계산되었습니다.
- priority_rule_detail은 rule_priority_hint가 부여된 규칙 설명입니다.
- priority_reason

# 8. PydanticAI Agent 설정

In [12]:
# ============================================================
# 패치·운영 전략 생성 Agent
# ============================================================

system_prompt = """
당신은 Steam 인디게임의 출시 후 패치·운영 전략을 정리하는 데이터 분석 보조자입니다.

당신의 역할은 우선 검토 수준 판단이 아니라 문장화와 전략 초안 작성입니다.
제공된 근거표의 action_group_hint와 rule_priority_hint를 반드시 그대로 사용하세요.
action_group_hint를 바꾸거나, rule_priority_hint를 새로 계산하거나, 올리거나, 내리지 마세요.
근거표에 없는 issue_name_kor를 새로 만들지 마세요.
issue_name에는 근거표의 issue_name_kor 값을 그대로 작성하세요.
priority_rule_detail과 priority_reason은 04-1에서 계산된 규칙 기반 근거입니다.
High urgency는 이전 LLM 리뷰 분류 결과를 집계한 보조 지표이므로, 단독 우선 검토 수준 기준으로 사용하지 마세요.
리뷰 수, 부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 이슈를 함께 언급하세요.
실제 원인이 확정된 것처럼 단정하지 말고, 패치와 운영에서 확인해야 할 방향으로 표현하세요.
"""

patch_ops_settings = GoogleModelSettings(
    temperature=TEMPERATURE,
)

if vertex_model is not None:
    patch_ops_agent = Agent(
        vertex_model,
        output_type=PostLaunchPatchOpsResult,
        system_prompt=system_prompt,
        retries=MAX_RETRIES,
        output_retries=3,
    )
else:
    patch_ops_agent = None

print("PydanticAI Agent 생성 여부:", "O" if patch_ops_agent is not None else "X")


PydanticAI Agent 생성 여부: O


# 9. LLM 호출 전 확인

In [13]:

# ============================================================
# LLM 호출 전 확인용 요약
# ============================================================
# RUN_PATCH_OPS_LLM=False 상태에서도 근거 데이터와 프롬프트가 잘 잡혔는지 확인한다.

print("분석 대상:", analysis_overview["game_name"])
print("분석 리뷰 수:", analysis_overview["review_count"])
print("분석 리뷰 기간:", analysis_overview["review_date_min"], "~", analysis_overview["review_date_max"])
print("Steam 라벨 분포:", analysis_overview["steam_label_distribution"])
print("LLM 감정 분포:", analysis_overview["llm_sentiment_distribution"])
print("High urgency 리뷰 수:", analysis_overview["high_urgency_review_count"])
print()

check_cols = [
    "issue_name_kor",
    "action_group_hint",
    "rule_priority_hint",
    "affected_review_count",
    "negative_mixed_review_count",
    "high_urgency_review_count",
    "recent_30d_negative_mixed_review_count",
    "early_playtime_negative_mixed_review_count",
]
check_cols = [col for col in check_cols if col in selected_evidence.columns]

print("선택된 근거 데이터")
print(selected_evidence[check_cols].to_string(index=False))
print()

print("프롬프트 앞부분")
print(patch_ops_prompt[:2500])

분석 대상: Tainted Grail: The Fall of Avalon
분석 리뷰 수: 1000
분석 리뷰 기간: 2026-01-29 ~ 2026-04-29
Steam 라벨 분포: positive: 671, negative: 329
LLM 감정 분포: positive: 592, negative: 302, mixed: 103, neutral: 3
High urgency 리뷰 수: 178

선택된 근거 데이터
issue_name_kor action_group_hint rule_priority_hint  affected_review_count  negative_mixed_review_count  high_urgency_review_count  recent_30d_negative_mixed_review_count  early_playtime_negative_mixed_review_count
            버그             즉시 확인                  상                     95                           85                         38                                      33                                          11
            성능             즉시 확인                  상                     60                           53                         26                                      22                                          13
           크래시             즉시 확인                  상                     30                           29                      

# 10. LLM 패치·운영 전략 생성

In [14]:
# ============================================================
# LLM 패치·운영 전략 생성
# ============================================================
# RUN_PATCH_OPS_LLM=False이면 실제 LLM 호출은 하지 않는다.

if RUN_PATCH_OPS_LLM:
    if patch_ops_agent is None:
        raise RuntimeError(
            "patch_ops_agent가 생성되지 않았습니다. "
            ".env의 GOOGLE_CLOUD_PROJECT, gcloud ADC 인증, pydantic-ai 설치 여부를 확인하세요."
        )

    result = await patch_ops_agent.run(
        patch_ops_prompt,
        model_settings=patch_ops_settings,
    )

    patch_ops_output = result.output
    patch_ops_result_dict = to_serializable(patch_ops_output)

    print("LLM 패치·운영 전략 생성 완료")

else:
    patch_ops_output = None
    patch_ops_result_dict = None

    print("RUN_PATCH_OPS_LLM=False")
    print("LLM 호출은 하지 않고, 프롬프트와 근거 데이터만 생성했습니다.")

LLM 패치·운영 전략 생성 완료


# 11. LLM 결과 검증 및 표 생성

In [15]:
# ============================================================
# LLM 결과 검증 및 표 생성
# ============================================================
# 목적:
# LLM이 action_group 또는 priority를 잘못 옮기더라도,
# 최종 출력에서는 04-1 근거표의 action_group_hint, rule_priority_hint를 다시 적용한다.


def build_fixed_maps(evidence_df):
    # 현재는 단일 게임 분석이므로 issue_name_kor 기준으로 고정값을 매핑한다.
    # 여러 게임을 동시에 다룰 경우 llm_issue_category까지 함께 키로 쓰는 방식이 더 안전하다.
    work = evidence_df.copy()

    priority_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["rule_priority_hint"]
        .to_dict()
    )

    action_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["action_group_hint"]
        .to_dict()
    )

    evidence_summary_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["priority_reason"]
        .to_dict()
    )

    rule_detail_map = (
        work
        .drop_duplicates("issue_name_kor")
        .set_index("issue_name_kor")["priority_rule_detail"]
        .to_dict()
        if "priority_rule_detail" in work.columns else {}
    )

    return priority_map, action_map, evidence_summary_map, rule_detail_map


priority_map, action_map, evidence_summary_map, rule_detail_map = build_fixed_maps(selected_evidence)


def validate_patch_ops_result(result_dict, evidence_df):
    priority_map, action_map, _, _ = build_fixed_maps(evidence_df)

    item_keys = [
        "immediate_actions",
        "short_term_improvements",
        "operation_communication",
        "long_term_reviews",
        "strengths_to_keep",
    ]

    warnings = []

    for key in item_keys:
        for item in result_dict.get(key, []):
            issue_name = item.get("issue_name", "")
            item_priority = item.get("priority", "")
            item_action_group = item.get("action_group", "")

            fixed_priority = priority_map.get(issue_name)
            fixed_action = action_map.get(issue_name)

            if fixed_priority is None or fixed_action is None:
                warnings.append(f"근거표에 없는 이슈가 LLM 결과에 포함됨: {issue_name}")
                continue

            if item_priority != fixed_priority:
                warnings.append(
                    f"이슈 '{issue_name}'의 priority={item_priority}, 근거표 rule_priority_hint={fixed_priority}"
                )

            if item_action_group != fixed_action:
                warnings.append(
                    f"이슈 '{issue_name}'의 action_group={item_action_group}, 근거표 action_group_hint={fixed_action}"
                )

    return warnings


def make_patch_ops_strategy_table(result_dict, evidence_df, drop_unknown_issues=True):
    priority_map, action_map, evidence_summary_map, rule_detail_map = build_fixed_maps(evidence_df)

    rows = []
    item_keys = [
        "immediate_actions",
        "short_term_improvements",
        "operation_communication",
        "long_term_reviews",
        "strengths_to_keep",
    ]

    for key in item_keys:
        for item in result_dict.get(key, []):
            issue_name = item.get("issue_name", "")

            if issue_name not in priority_map:
                if drop_unknown_issues:
                    continue
                fixed_priority = item.get("priority", "")
                fixed_action = item.get("action_group", "")
            else:
                fixed_priority = priority_map[issue_name]
                fixed_action = action_map[issue_name]

            detailed_actions = item.get("detailed_actions", [])
            if isinstance(detailed_actions, list):
                detailed_actions_text = "\n".join([f"- {x}" for x in detailed_actions])
            else:
                detailed_actions_text = str(detailed_actions)

            rows.append({
                "대응 구분": fixed_action,
                "우선 검토 수준": fixed_priority,
                "이슈": issue_name,
                "패치·운영 방향": item.get("patch_ops_direction", ""),
                "세부 실행안": detailed_actions_text,
                "근거 요약": item.get("evidence_summary", evidence_summary_map.get(issue_name, "")),
                "규칙 근거": rule_detail_map.get(issue_name, ""),
                "기대 효과": item.get("expected_effect", ""),
                "주의사항": item.get("caution", ""),
            })

    strategy_df = pd.DataFrame(rows)

    if len(strategy_df) == 0:
        return strategy_df

    priority_order_map = {"상": 1, "중": 2, "하": 3}
    action_order_map = {
        "즉시 확인": 1,
        "단기 개선": 2,
        "운영 커뮤니케이션 개선": 3,
        "장기 검토": 4,
        "검토 필요": 5,
        "강점 유지": 6,
    }

    strategy_df["action_order"] = strategy_df["대응 구분"].map(action_order_map).fillna(9)
    strategy_df["priority_order"] = strategy_df["우선 검토 수준"].map(priority_order_map).fillna(9)
    strategy_df = strategy_df.sort_values(["action_order", "priority_order", "이슈"])
    strategy_df = strategy_df.drop(columns=["action_order", "priority_order"])

    return strategy_df


if patch_ops_result_dict is not None:
    validation_warnings = validate_patch_ops_result(
        patch_ops_result_dict,
        evidence_df=selected_evidence,
    )

    patch_ops_strategy_df = make_patch_ops_strategy_table(
        patch_ops_result_dict,
        evidence_df=selected_evidence,
        drop_unknown_issues=True,
    )
else:
    validation_warnings = []
    patch_ops_strategy_df = pd.DataFrame()

print("LLM 출력 검증 메시지 수:", len(validation_warnings))
print("패치·운영 전략 표 행 수:", len(patch_ops_strategy_df))

LLM 출력 검증 메시지 수: 0
패치·운영 전략 표 행 수: 18


# 12. 최종 출력 공통 함수

In [16]:
# ============================================================
# 공통 출력 함수
# ============================================================
# 목적:
# table-version과 report-version에서 동일한 순서와 같은 형식으로
# 분석 개요, 우선 검토 수준 기준, 이슈별 근거표, LLM 출력 검증 결과를 보여준다.


def display_block_title(title):
    display(Markdown(f"## {title}"))


def make_analysis_overview_df(analysis_overview):
    rows = [
        ("대상 게임", analysis_overview.get("game_name", "")),
        ("appid", analysis_overview.get("appid", "")),
        ("분석 리뷰 수", f"{analysis_overview.get('review_count', 0):,}개"),
        ("분석 이슈 태그 수", f"{analysis_overview.get('issue_tag_count', 0):,}개"),
        ("분석 리뷰 기간", f"{analysis_overview.get('review_date_min', '')} ~ {analysis_overview.get('review_date_max', '')}"),
        ("Steam 라벨 분포", analysis_overview.get("steam_label_distribution", "")),
        ("LLM 감정 분포", analysis_overview.get("llm_sentiment_distribution", "")),
        ("High urgency 리뷰 수", f"{analysis_overview.get('high_urgency_review_count', 0):,}개"),
        ("최근 30일 리뷰 수", f"{analysis_overview.get('recent_30d_review_count', 0):,}개"),
    ]
    return pd.DataFrame(rows, columns=["항목", "값"])


def make_priority_criteria_df():
    rows = [
        {
            "항목": "우선 검토 수준 산정 주체",
            "기준": "LLM이 직접 판단하지 않고, 04-1에서 계산한 rule_priority_hint를 그대로 사용한다.",
        },
        {
            "항목": "핵심 근거",
            "기준": "부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 부정·혼합 반응을 사용한다.",
        },
        {
            "항목": "High urgency 사용 방식",
            "기준": "04번 리뷰 분류 단계에서 LLM이 분류한 보조 지표이며, 단독으로 우선 검토 수준을 올리는 기준으로 사용하지 않는다.",
        },
        {
            "항목": "04-2에서 LLM의 역할",
            "기준": "이미 계산된 우선 검토 수준과 대응 구분을 바탕으로 개발자가 이해하기 쉬운 문장과 실행안으로 정리한다.",
        },
    ]
    return pd.DataFrame(rows)


def make_evidence_display_df(evidence_df, max_rows=20):
    display_cols = [
        "issue_name_kor",
        "action_group_hint",
        "rule_priority_hint",
        "priority_rule_detail",
        "affected_review_count",
        "negative_mixed_review_count",
        "steam_negative_review_count",
        "high_urgency_review_count",
        "recent_30d_negative_mixed_review_count",
        "early_playtime_negative_mixed_review_count",
        "priority_reason",
    ]
    display_cols = [col for col in display_cols if col in evidence_df.columns]

    rename_map = {
        "issue_name_kor": "이슈",
        "action_group_hint": "대응 구분",
        "rule_priority_hint": "우선 검토 수준",
        "priority_rule_detail": "규칙 근거",
        "affected_review_count": "영향 리뷰 수",
        "negative_mixed_review_count": "부정·혼합 리뷰 수",
        "steam_negative_review_count": "Steam 비추천 리뷰 수",
        "high_urgency_review_count": "High urgency 리뷰 수",
        "recent_30d_negative_mixed_review_count": "최근 30일 부정·혼합",
        "early_playtime_negative_mixed_review_count": "짧은 플레이타임 부정·혼합",
        "priority_reason": "근거 요약",
    }

    return evidence_df[display_cols].head(max_rows).rename(columns=rename_map)


def make_validation_result_df(validation_warnings):
    if len(validation_warnings) == 0:
        return pd.DataFrame([
            {
                "검증 항목": "LLM 출력 검증",
                "결과": "통과",
                "내용": "LLM이 출력한 이슈의 대응 구분과 우선 검토 수준이 04-1 근거표 기준과 일치한다.",
            }
        ])

    return pd.DataFrame([
        {
            "검증 항목": "LLM 출력 검증",
            "결과": "보정 필요",
            "내용": warning,
        }
        for warning in validation_warnings
    ])


def display_common_output_blocks():
    display_block_title("1. 분석 개요")
    display(make_analysis_overview_df(analysis_overview))

    display_block_title("2. 우선순위 판단 기준")
    display(make_priority_criteria_df())

    display_block_title("3. 04-1에서 계산한 이슈별 근거표")
    display(make_evidence_display_df(selected_evidence))

    display_block_title("4. LLM 출력 검증 결과")
    display(make_validation_result_df(validation_warnings))


print("공통 출력 함수 정의 완료")

공통 출력 함수 정의 완료


# 13. 리포트 형식 패치·운영 전략 출력

In [17]:
# ============================================================
# 리포트 형식 패치·운영 전략 출력
# ============================================================
# 목적:
# - report-version도 table-version과 동일하게 분석 개요, 우선순위 기준, 근거표, 검증 결과를 먼저 출력한다.
# - 마지막 전략 제안만 발표/보고서에 붙여넣기 쉬운 리포트 문장 형식으로 보여준다.
# - 파일 저장 없이 코드 화면에만 출력한다.


def _split_multiline_actions(text):
    if not isinstance(text, str) or not text.strip():
        return []
    actions = []
    for line in text.split("\n"):
        clean = line.strip().lstrip("- ").strip()
        if clean:
            actions.append(clean)
    return actions


def build_patch_ops_report_markdown(result_dict, strategy_df):
    if result_dict is None or strategy_df is None or len(strategy_df) == 0:
        return "## 5. 패치·운영 전략 제안 리포트\n\nLLM 결과가 없어 패치·운영 전략 제안 리포트를 출력하지 못했다."

    lines = []
    lines.append("## 5. 패치·운영 전략 제안 리포트")
    lines.append("")
    lines.append(f"### {result_dict.get('title', '출시 후 패치·운영 전략 제안')}")
    lines.append("")

    summary_items = [
        ("게임 요약", result_dict.get("game_summary", "")),
        ("데이터 요약", result_dict.get("data_summary", "")),
        ("현재 반응 상태", result_dict.get("current_status_summary", "")),
    ]

    for subtitle, value in summary_items:
        if value:
            lines.append(f"#### {subtitle}")
            lines.append(value)
            lines.append("")

    action_order = [
        ("즉시 확인", "지금 먼저 확인할 일"),
        ("단기 개선", "다음 업데이트에서 개선할 일"),
        ("운영 커뮤니케이션 개선", "운영 커뮤니케이션으로 보완할 일"),
        ("장기 검토", "장기적으로 보강할 일"),
        ("검토 필요", "추가 확인이 필요한 일"),
        ("강점 유지", "계속 살릴 강점"),
    ]

    for action_group, section_title in action_order:
        group_df = strategy_df[strategy_df["대응 구분"] == action_group].copy()
        if len(group_df) == 0:
            continue

        lines.append(f"### {section_title}")
        lines.append("")

        for _, row in group_df.iterrows():
            issue = row.get("이슈", "")
            priority = row.get("우선순위", "")
            direction = str(row.get("패치·운영 방향", "")).strip()
            evidence = str(row.get("근거 요약", "")).strip()
            expected = str(row.get("기대 효과", "")).strip()
            caution = str(row.get("주의사항", "")).strip()
            actions = _split_multiline_actions(row.get("세부 실행안", ""))

            lines.append(f"#### {issue} / 우선순위 {priority}")
            if evidence:
                lines.append(f"- 근거: {evidence}")
            if direction:
                lines.append(f"- 제안 방향: {direction}")
            if actions:
                lines.append("- 세부 실행안:")
                for action in actions:
                    lines.append(f"  - {action}")
            if expected:
                lines.append(f"- 기대 효과: {expected}")
            if caution:
                lines.append(f"- 주의사항: {caution}")
            lines.append("")

    operation_notes = result_dict.get("operation_notes", [])
    if operation_notes:
        lines.append("### 운영 관점 제안")
        for note in operation_notes:
            lines.append(f"- {note}")
        lines.append("")

    cautions = result_dict.get("cautions", [])
    if cautions:
        lines.append("### 해석 시 주의사항")
        for caution in cautions:
            lines.append(f"- {caution}")
        lines.append("")

    final_summary = result_dict.get("final_summary", "")
    if final_summary:
        lines.append("### 최종 요약")
        lines.append(final_summary)
        lines.append("")

    return "\n".join(lines)


display_common_output_blocks()
report_markdown = build_patch_ops_report_markdown(patch_ops_result_dict, patch_ops_strategy_df)
display(Markdown(report_markdown))

## 1. 분석 개요

,항목,값
0,대상 게임,Tainted Grail: The Fall of Avalon
1,appid,1466060
2,분석 리뷰 수,"1,000개"
3,분석 이슈 태그 수,"1,967개"
4,분석 리뷰 기간,2026-01-29 ~ 2026-04-29
5,Steam 라벨 분포,"positive: 671, negative: 329"
6,LLM 감정 분포,"positive: 592, negative: 302, mixed: 103, neutral: 3"
7,High urgency 리뷰 수,178개
8,최근 30일 리뷰 수,373개


## 2. 우선순위 판단 기준

,항목,기준
0,우선 검토 수준 산정 주체,"LLM이 직접 판단하지 않고, 04-1에서 계산한 rule_priority_hint를 그대로 사용한다."
1,핵심 근거,"부정·혼합 리뷰 수, Steam 비추천 리뷰 수, 최근 30일 반복 여부, 짧은 플레이타임 부정·혼합 반응을 사용한다."
2,High urgency 사용 방식,"04번 리뷰 분류 단계에서 LLM이 분류한 보조 지표이며, 단독으로 우선 검토 수준을 올리는 기준으로 사용하지 않는다."
3,04-2에서 LLM의 역할,이미 계산된 우선 검토 수준과 대응 구분을 바탕으로 개발자가 이해하기 쉬운 문장과 실행안으로 정리한다.


## 3. 04-1에서 계산한 이슈별 근거표

,이슈,대응 구분,우선 검토 수준,규칙 근거,영향 리뷰 수,부정·혼합 리뷰 수,Steam 비추천 리뷰 수,High urgency 리뷰 수,최근 30일 부정·혼합,짧은 플레이타임 부정·혼합,근거 요약
0,버그,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,95,85,41,38,33,11,"영향 리뷰 95개, 부정·혼합 85개, Steam 비추천 맥락 41개, 최근 30일 부정·혼합 33개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 38개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
1,성능,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,60,53,26,26,22,13,"영향 리뷰 60개, 부정·혼합 53개, Steam 비추천 맥락 26개, 최근 30일 부정·혼합 22개, 초반 플레이타임 부정·혼합 13개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
2,크래시,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,30,29,25,26,11,7,"영향 리뷰 30개, 부정·혼합 29개, Steam 비추천 맥락 25개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 7개, High urgency 후보 26개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
3,최적화,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,27,26,16,14,11,11,"영향 리뷰 27개, 부정·혼합 26개, Steam 비추천 맥락 16개, 최근 30일 부정·혼합 11개, 초반 플레이타임 부정·혼합 11개, High urgency 후보 14개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
4,저장/진행,즉시 확인,상,플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락,27,24,23,23,8,3,"영향 리뷰 27개, 부정·혼합 24개, Steam 비추천 맥락 23개, 최근 30일 부정·혼합 8개, 초반 플레이타임 부정·혼합 3개, High urgency 후보 23개(보조 참고), 규칙 근거: 플레이 방해 가능 이슈 + 부정·혼합 반복 + Steam 비추천 맥락"
5,게임플레이 루프,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,284,208,150,73,81,41,"영향 리뷰 284개, 부정·혼합 208개, Steam 비추천 맥락 150개, 최근 30일 부정·혼합 81개, 초반 플레이타임 부정·혼합 41개, High urgency 후보 73개(보조 참고), 규칙 근거: 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"
6,밸런스,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,130,118,64,40,50,6,"영향 리뷰 130개, 부정·혼합 118개, Steam 비추천 맥락 64개, 최근 30일 부정·혼합 50개, 초반 플레이타임 부정·혼합 6개, High urgency 후보 40개(보조 참고), 규칙 근거: 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"
7,UI/UX,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,63,58,36,20,19,8,"영향 리뷰 63개, 부정·혼합 58개, Steam 비추천 맥락 36개, 최근 30일 부정·혼합 19개, 초반 플레이타임 부정·혼합 8개, High urgency 후보 20개(보조 참고), 규칙 근거: 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"
8,난이도,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,68,48,30,19,21,7,"영향 리뷰 68개, 부정·혼합 48개, Steam 비추천 맥락 30개, 최근 30일 부정·혼합 21개, 초반 플레이타임 부정·혼합 7개, High urgency 후보 19개(보조 참고), 규칙 근거: 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"
9,성장/반복 노가다,단기 개선,상,부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복,28,26,16,14,9,2,"영향 리뷰 28개, 부정·혼합 26개, Steam 비추천 맥락 16개, 최근 30일 부정·혼합 9개, 초반 플레이타임 부정·혼합 2개, High urgency 후보 14개(보조 참고), 규칙 근거: 부정·혼합 반복 + Steam 비추천 맥락 + 최근 30일 반복"


## 4. LLM 출력 검증 결과

,검증 항목,결과,내용
0,LLM 출력 검증,통과,LLM이 출력한 이슈의 대응 구분과 우선 검토 수준이 04-1 근거표 기준과 일치한다.


## 5. 패치·운영 전략 제안 리포트

### Tainted Grail: The Fall of Avalon 출시 후 패치·운영 전략 보고서

#### 게임 요약
Tainted Grail: The Fall of Avalon은 다크 판타지 세계관을 배경으로 한 오픈 월드 RPG로, 현재 기술적 안정성과 게임플레이 완성도 측면에서 유저들의 개선 요구가 높은 상태입니다.

#### 데이터 요약
분석 대상은 'Tainted Grail: The Fall of Avalon'(AppID: 1466060)이며, 2026년 1월 29일부터 4월 29일까지의 리뷰 1,000건과 이슈 태그 1,967건을 분석하였습니다.

#### 현재 반응 상태
전체 1,000개의 리뷰 중 긍정적인 반응이 671개로 과반을 차지하나, 최근 30일간 373개의 리뷰가 집중되는 등 기술적 결함과 게임플레이 루프에 대한 부정적 피드백이 지속적으로 유입되고 있습니다. 특히 버그, 성능, 크래시 등 플레이를 직접적으로 방해하는 이슈가 Steam 비추천의 주요 원인으로 작용하고 있습니다.

### 지금 먼저 확인할 일

#### 버그 / 우선순위 
- 근거: 영향 리뷰 95개, 부정·혼합 85개, Steam 비추천 41개, 최근 30일 부정·혼합 33개로 플레이 방해 이슈가 반복됨.
- 제안 방향: 반복 언급된 버그를 재현 가능성 기준으로 분류하고, 플레이 방해 수준이 큰 항목부터 수정한다.
- 세부 실행안:
  - 반복 언급된 퀘스트 NPC 스폰 버그 재현 및 수정
  - 주요 분기점의 퀘스트 로직 검토 및 대체 경로 마련
  - 소프트 락 유발 버그 수정 및 세이브 데이터 손실 방지 조치
- 기대 효과: 진행 불가 현상 해소를 통한 초반 이탈률 감소 및 유저 경험 안정화.
- 주의사항: 재현 경로가 불분명한 경우 유저의 시스템 사양과 로그를 우선 확보해야 합니다.

#### 성능 / 우선순위 
- 근거: 영향 리뷰 60개, 부정·혼합 53개, Steam 비추천 26개, 최근 30일 부정·혼합 22개로 성능 이슈가 지속됨.
- 제안 방향: 프레임 저하, 로딩, 끊김 등 성능 문제를 환경별로 점검한다.
- 세부 실행안:
  - 스팀 덱 및 저사양 환경 최적화 패치 우선순위 상향
  - 오픈 월드 구간의 리소스 로딩 및 렌더링 최적화
  - 프레임 드랍 및 스터터링 현상 개선 패치 배포
- 기대 효과: 프레임 안정화를 통한 전반적인 플레이 쾌적도 향상.
- 주의사항: 특정 하드웨어 환경(스팀 덱 등)에서의 성능 저하를 우선적으로 고려해야 합니다.

#### 저장/진행 / 우선순위 
- 근거: 영향 리뷰 27개, 부정·혼합 24개, Steam 비추천 23개, 최근 30일 부정·혼합 8개로 저장/진행 이슈가 심각함.
- 제안 방향: 저장 손실, 진행 막힘, 퀘스트 진행 불가 여부를 우선 점검한다.
- 세부 실행안:
  - 세이브 데이터 손상 원인 파악을 위한 긴급 로그 분석
  - 진행 불가능 상태(soft-lock) 유발 퀘스트 로직 수정
  - 세이브 파일 백업 시스템 강화 및 자동 세이브 슬롯 추가 검토
- 기대 효과: 데이터 손실 방지 및 게임 진행 안정성 확보.
- 주의사항: 세이브 데이터 손상은 유저에게 가장 치명적이므로 백업 시스템 강화가 필수적입니다.

#### 최적화 / 우선순위 
- 근거: 영향 리뷰 27개, 부정·혼합 26개, Steam 비추천 16개, 최근 30일 부정·혼합 11개로 최적화 불만이 확인됨.
- 제안 방향: 최적화 이슈가 특정 구간이나 사양에서 반복되는지 확인한다.
- 세부 실행안:
  - 오픈 월드 텍스처 스트리밍 및 LOD 설정 최적화
  - 초기 실행 시 블랙 스크린 및 응답 없음 문제 호환성 체크
  - 특정 사양에서의 화면 깜빡임 이슈 기술적 원인 조사
- 기대 효과: 기술적 완성도 향상을 통한 유저 신뢰도 회복.
- 주의사항: 그래픽 설정과 최적화 간의 균형을 맞추는 작업이 필요합니다.

#### 크래시 / 우선순위 
- 근거: 영향 리뷰 30개, 부정·혼합 29개, Steam 비추천 25개, 최근 30일 부정·혼합 11개로 크래시가 빈번함.
- 제안 방향: 크래시 발생 조건과 로그를 우선 확인하고, 재현 가능한 오류부터 수정한다.
- 세부 실행안:
  - 크래시 리포트 로그 분석 및 메모리 누수 원인 파악
  - 특정 지역(Cuanacht 등) 진입 시 발생하는 크래시 핫픽스 배포
  - 자동 저장 기능과 관련된 기술적 결함 수정
- 기대 효과: 게임 강제 종료 현상 감소를 통한 플레이 지속성 확보.
- 주의사항: 크래시 로그 분석 시 메모리 누수 여부를 정밀하게 확인해야 합니다.

### 다음 업데이트에서 개선할 일

#### UI/UX / 우선순위 
- 근거: 영향 리뷰 63개, 부정·혼합 58개, Steam 비추천 36개, 최근 30일 부정·혼합 19개로 UI/UX 불편함이 확인됨.
- 제안 방향: 메뉴, 인벤토리, 퀘스트 안내, 조작 안내 등 편의성 문제를 개선한다.
- 세부 실행안:
  - 메뉴, 인벤토리, 퀘스트 안내 등 편의성 문제 개선
  - 마법 시전 UI 개선 및 단축키 기능 추가 검토
- 기대 효과: 사용자 편의성 증대 및 조작 스트레스 감소.
- 주의사항: UI/UX 개선은 기존 유저의 조작 습관을 고려하여 점진적으로 적용해야 합니다.

#### 게임플레이 루프 / 우선순위 
- 근거: 영향 리뷰 284개, 부정·혼합 208개, Steam 비추천 150개, 최근 30일 부정·혼합 81개로 게임플레이 루프에 대한 불만이 가장 큼.
- 제안 방향: 반복 피로, 목표 구조, 보상 흐름을 점검하고 플레이 루프의 지루함을 줄인다.
- 세부 실행안:
  - 반복 피로 감소를 위한 목표 구조 및 보상 흐름 점검
  - 플레이 루프의 지루함을 줄이기 위한 콘텐츠 배치 조정
- 기대 효과: 게임의 핵심 재미 요소 강화 및 플레이 지속성 향상.
- 주의사항: 플레이 루프 개선 시 기존 유저의 숙련도를 고려해야 합니다.

#### 난이도 / 우선순위 
- 근거: 영향 리뷰 68개, 부정·혼합 48개, Steam 비추천 30개, 최근 30일 부정·혼합 21개로 난이도에 대한 불만이 있음.
- 제안 방향: 초반 진입 장벽과 후반 난이도 피로를 구분해 난이도 옵션 또는 안내를 보강한다.
- 세부 실행안:
  - 초반 진입 장벽 완화 및 후반 난이도 피로도 개선
  - 난이도 선택 옵션 추가 또는 안내 보강 검토
- 기대 효과: 다양한 유저층 확보 및 진입 장벽 완화.
- 주의사항: 난이도 옵션 추가 시 게임의 본래 의도된 난이도를 훼손하지 않도록 주의해야 합니다.

#### 밸런스 / 우선순위 
- 근거: 영향 리뷰 130개, 부정·혼합 118개, Steam 비추천 64개, 최근 30일 부정·혼합 50개로 밸런스 이슈가 반복됨.
- 제안 방향: 전투, 성장, 보상, 적 난이도의 불균형 지점을 조정한다.
- 세부 실행안:
  - 전투, 성장, 보상, 적 난이도 불균형 지점 조정
  - 마법 빌드와 타 빌드 간의 밸런스 재조정
- 기대 효과: 전투 긴장감 회복 및 빌드 다양성 확보.
- 주의사항: 밸런스 조정은 특정 빌드만 강력해지지 않도록 다각도로 검토해야 합니다.

#### 성장/반복 노가다 / 우선순위 
- 근거: 영향 리뷰 28개, 부정·혼합 26개, Steam 비추천 16개, 최근 30일 부정·혼합 9개로 성장/반복 노가다에 대한 불만이 있음.
- 제안 방향: 반복 성장과 노가다 피로를 줄일 수 있는 보상/성장 속도 조정을 검토한다.
- 세부 실행안:
  - 반복 성장 및 노가다 피로 감소를 위한 보상/성장 속도 조정
  - 장비 업그레이드 비용 및 파밍 효율 개선 검토
- 기대 효과: 성장 체감 향상 및 반복 플레이 피로도 감소.
- 주의사항: 성장 속도 조정 시 게임의 전체적인 플레이 타임에 미치는 영향을 고려해야 합니다.

#### 조작감 / 우선순위 
- 근거: 영향 리뷰 19개, 부정·혼합 19개, Steam 비추천 12개, 최근 30일 부정·혼합 6개로 조작감에 대한 불만이 있음.
- 제안 방향: 이동, 전투, 상호작용 조작의 반응성과 키 설정 편의성을 점검한다.
- 세부 실행안:
  - 이동, 전투, 상호작용 조작 반응성 및 키 설정 편의성 점검
  - 캐릭터 이동 및 조작감 개선을 위한 애니메이션 업데이트 검토
- 기대 효과: 전투 및 이동 조작의 쾌적함 향상.
- 주의사항: 조작감 개선은 애니메이션과 입력 반응성 간의 조화가 중요합니다.

### 운영 커뮤니케이션으로 보완할 일

#### 개발사 소통 / 우선순위 
- 근거: 영향 리뷰 13개, 부정·혼합 7개, Steam 비추천 7개, 최근 30일 부정·혼합 2개로 소통에 대한 개선 요구가 있음.
- 제안 방향: 패치 노트, 공지, 커뮤니티 응답 등 운영 커뮤니케이션을 점검한다.
- 세부 실행안:
  - 패치 노트 및 알려진 이슈 안내 강화
  - 커뮤니티 피드백에 대한 투명한 소통 창구 마련 및 대응 강화
  - 민감한 소재 및 운영 정책에 대한 공식 입장 표명
- 기대 효과: 개발사-유저 간 신뢰 회복 및 커뮤니티 분위기 개선.
- 주의사항: 민감한 소재에 대한 소통은 투명하고 유연하게 대응해야 합니다.

### 장기적으로 보강할 일

#### 가격/가치 / 우선순위 
- 근거: 영향 리뷰 26개, 부정·혼합 20개, Steam 비추천 20개, 최근 30일 부정·혼합 11개로 가격/가치에 대한 불만이 있음.
- 제안 방향: 가격 대비 만족도 불만이 반복되는지 확인하고 할인/번들/콘텐츠 가치 전달을 검토한다.
- 세부 실행안:
  - 가격 대비 만족도 불만 원인 분석 및 콘텐츠 가치 전달 강화
  - 할인/번들 구성 및 향후 업데이트 가치 홍보 검토
- 기대 효과: 가격 대비 만족도 제고 및 구매 전환율 개선.
- 주의사항: 가격 정책 변경은 기존 구매자와의 형평성을 고려해야 합니다.

#### 그래픽/사운드 / 우선순위 
- 근거: 영향 리뷰 133개, 부정·혼합 95개, Steam 비추천 59개, 최근 30일 부정·혼합 38개로 그래픽/사운드에 대한 개선 요구가 높음.
- 제안 방향: 그래픽/사운드가 몰입을 방해하는지와 강점으로 작동하는지를 함께 확인한다.
- 세부 실행안:
  - 주요 NPC 모델링 텍스처 및 애니메이션 개선 검토
  - 몰입감을 저해하는 그래픽 요소 및 사운드 연출 보강
- 기대 효과: 시각적/청각적 완성도 향상을 통한 몰입감 증대.
- 주의사항: 그래픽/사운드 개선은 게임의 전체적인 분위기를 해치지 않는 선에서 진행해야 합니다.

#### 스토리 / 우선순위 
- 근거: 영향 리뷰 153개, 부정·혼합 89개, Steam 비추천 65개, 최근 30일 부정·혼합 31개로 스토리 만족도가 낮음.
- 제안 방향: 서사 전달, 퀘스트 흐름, 엔딩/분기 만족도를 장기 개선 후보로 검토한다.
- 세부 실행안:
  - 주요 스토리 분기점 연출 보강 및 서사 전달력 개선
  - 퀘스트 흐름 및 엔딩 만족도 분석을 통한 장기 개선 로드맵 수립
- 기대 효과: 서사적 몰입감 향상 및 스토리 만족도 개선.
- 주의사항: 스토리 구조 변경은 기존 세이브 데이터와의 호환성을 고려해야 합니다.

#### 콘텐츠 분량 / 우선순위 
- 근거: 영향 리뷰 92개, 부정·혼합 64개, Steam 비추천 34개, 최근 30일 부정·혼합 24개로 콘텐츠 분량에 대한 불만이 확인됨.
- 제안 방향: 콘텐츠 부족·반복성은 단기 패치보다 업데이트 로드맵 관점에서 검토한다.
- 세부 실행안:
  - 콘텐츠 부족 및 반복성 해소를 위한 업데이트 로드맵 공유
  - 중후반부 콘텐츠 밀도 보강 검토
- 기대 효과: 플레이 타임 연장 및 콘텐츠 만족도 향상.
- 주의사항: 콘텐츠 추가는 게임의 전체적인 밸런스를 고려하여 신중하게 진행해야 합니다.

### 추가 확인이 필요한 일

#### 기타 / 우선순위 
- 근거: 영향 리뷰 99개, 부정·혼합 70개, Steam 비추천 63개, 최근 30일 부정·혼합 23개로 기타 이슈가 다수 확인됨.
- 제안 방향: 세부 리뷰를 확인해 반복되는 하위 원인이 있는지 검토한다.
- 세부 실행안:
  - 반복되는 하위 원인 분석을 위한 세부 리뷰 전수 조사
  - 유저 피드백 기반의 우선순위 재조정 검토
- 기대 효과: 잠재적 문제 조기 발견 및 대응.
- 주의사항: 기타 이슈는 세부 리뷰를 면밀히 분석하여 새로운 기술적 결함이 있는지 확인해야 합니다.

### 계속 살릴 강점

#### 긍정 칭찬 / 우선순위 
- 근거: 영향 리뷰 615개로 긍정적인 평가가 압도적이며, 부정적 맥락은 거의 없음.
- 제안 방향: 긍정적으로 평가된 요소를 유지하고, 업데이트와 마케팅 메시지에서 강점으로 활용한다.
- 세부 실행안:
  - 긍정적으로 평가된 세계관 및 로어 요소 유지 및 강화
  - 업데이트 및 마케팅 메시지에 강점 요소 적극 활용
- 기대 효과: 기존 팬층 유지 및 신규 유저 유입 촉진.
- 주의사항: 긍정적인 요소가 업데이트 과정에서 훼손되지 않도록 주의해야 합니다.

### 운영 관점 제안
- 기술적 이슈(버그, 성능, 크래시)에 대한 패치 노트를 상세히 작성하고, 알려진 이슈(Known Issues) 목록을 커뮤니티에 상시 게시하여 유저 불안을 해소한다.
- 주요 업데이트 로드맵을 공유하여 콘텐츠 부족 및 최적화에 대한 유저들의 기대치를 관리한다.
- 커뮤니티 내 민감한 이슈에 대해서는 투명하고 신속하게 공식 입장을 밝혀 불필요한 오해를 방지한다.

### 해석 시 주의사항
- 리뷰 데이터는 유저의 주관적인 경험을 바탕으로 하므로, 기술적 이슈의 경우 반드시 내부 로그 및 재현 테스트를 통해 원인을 확정해야 합니다.
- High urgency 지표는 보조적인 참고 자료이며, 우선순위는 데이터 기반의 rule_priority_hint를 최우선으로 고려해야 합니다.
- 단기 개선 및 장기 검토 항목은 개발 리소스와 로드맵을 고려하여 단계적으로 접근해야 하며, 모든 이슈를 동시에 해결하려 하기보다 핵심적인 플레이 방해 요소를 우선해야 합니다.

### 최종 요약
현재 게임은 세계관과 로어에 대한 긍정적인 평가를 받고 있으나, 기술적 완성도 부족과 반복적인 게임플레이 루프가 성장을 저해하고 있습니다. 즉시 플레이 방해 요소를 제거하는 기술 패치를 최우선으로 진행하고, 이후 유저 피드백을 반영한 밸런스 조정 및 콘텐츠 개선 로드맵을 투명하게 공유하여 신뢰를 회복하는 운영 전략이 필요합니다.
